In [ ]:
# In[1]:

import pandas as pd

# Load CSVs (they are in the working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (df_metric, df_trace, df_log, df_error):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Prepare file-level compact summary
def file_summary(df, file_name):
    return {
        "file_name": file_name,
        "number_of_rows": int(df.shape[0]),
        "earliest_timestamp": df['timestamp'].min(),
        "latest_timestamp": df['timestamp'].max(),
        "number_of_unique_cmdb_id": int(df['cmdb_id'].nunique())
    }

summary_rows = [
    file_summary(df_metric, "metric.csv"),
    file_summary(df_trace, "trace.csv"),
    file_summary(df_log, "log.csv"),
    file_summary(df_error, "error_logs.csv"),
]
summary_df = pd.DataFrame(summary_rows)

# Helper to compute grouped counts and global P95 for a (cmdb_id, name_col) pair
def top_pairs_with_p95(df, id_col, name_col, value_col='value', top_n=20):
    # Ensure numeric values for percentile calculation
    if value_col in df.columns:
        df['_num_value'] = pd.to_numeric(df[value_col], errors='coerce')
    else:
        df['_num_value'] = pd.Series(dtype='float64')
    grp = df.groupby([id_col, name_col], dropna=False)
    agg = grp.agg(
        count_rows = ('timestamp', 'count'),
        p95_value = ('_num_value', lambda s: s.quantile(0.95) if s.notna().any() else float('nan'))
    ).reset_index()
    agg = agg.sort_values('count_rows', ascending=False).reset_index(drop=True)
    # rename name_col to a consistent column label for display
    agg = agg.rename(columns={name_col: 'name'})
    return agg[['cmdb_id', 'name', 'count_rows', 'p95_value']].rename(columns={'name': name_col}).head(top_n)

# Compute metric pairs (cmdb_id, kpi_name)
metric_pairs_df = top_pairs_with_p95(df_metric, 'cmdb_id', 'kpi_name', 'value', top_n=20)

# Compute trace pairs (cmdb_id, trace_name)
trace_pairs_df = top_pairs_with_p95(df_trace, 'cmdb_id', 'trace_name', 'value', top_n=20)

# Compute log pairs (cmdb_id, log_name)
log_pairs_df = top_pairs_with_p95(df_log, 'cmdb_id', 'log_name', 'value', top_n=20)

# Compute error_logs counts per cmdb_id (top 20)
error_counts_df = (
    df_error.groupby('cmdb_id', dropna=False)
    .agg(error_count=('timestamp', 'count'))
    .reset_index()
    .sort_values('error_count', ascending=False)
    .reset_index(drop=True)
)
error_counts_df = error_counts_df.head(20)

# Keep outputs compact: show summary and top lists (max 20 rows each)
summary_df, metric_pairs_df, trace_pairs_df, log_pairs_df, error_counts_df

```
Out[1]:
```
summary = (
    "Telemetry summary (timestamps UTC):\n\n"
    "File-level totals:\n"
    f"- metric.csv: {int(summary_df.loc[summary_df.file_name=='metric.csv','number_of_rows'].iloc[0])} rows; "
    f"earliest {summary_df.loc[summary_df.file_name=='metric.csv','earliest_timestamp'].iloc[0]}; "
    f"latest {summary_df.loc[summary_df.file_name=='metric.csv','latest_timestamp'].iloc[0]}; "
    f"{int(summary_df.loc[summary_df.file_name=='metric.csv','number_of_unique_cmdb_id'].iloc[0])} unique cmdb_id.\n"
    f"- trace.csv: {int(summary_df.loc[summary_df.file_name=='trace.csv','number_of_rows'].iloc[0])} rows (empty); "
    "no timestamps or cmdb_ids.\n"
    f"- log.csv: {int(summary_df.loc[summary_df.file_name=='log.csv','number_of_rows'].iloc[0])} rows; "
    f"earliest {summary_df.loc[summary_df.file_name=='log.csv','earliest_timestamp'].iloc[0]}; "
    f"latest {summary_df.loc[summary_df.file_name=='log.csv','latest_timestamp'].iloc[0]}; "
    f"{int(summary_df.loc[summary_df.file_name=='log.csv','number_of_unique_cmdb_id'].iloc[0])} unique cmdb_id.\n"
    f"- error_logs.csv: {int(summary_df.loc[summary_df.file_name=='error_logs.csv','number_of_rows'].iloc[0])} rows (empty).\n\n"
    "Top metric (cmdb_id, kpi_name) pairs (by row count, up to 20) with global P95 of value:\n"
    "- carts / cpu: count=25, p95=2.52205\n"
    "- carts / latency-50: count=25, p95=0.01976062\n"
    "- carts / latency-90: count=25, p95=0.04346478\n"
    "- carts / mem: count=25, p95=213711700.0\n"
    "- carts / socket: count=25, p95=12.0\n"
    "(Additional top metric pairs exist for carts-db, catalogue, catalogue-db, etc.; counts are mostly 25.)\n\n"
    "Trace data: no trace rows available.\n\n"
    "Top log (cmdb_id, log_name) pairs (by row count, up to 20) with global P95 of value:\n"
    "- front-end / log.row_count: count=25, p95=2074.2\n"
    "- front-end / log.error_count: count=25, p95=0.0\n"
    "- catalogue / log.row_count: count=25, p95=171.2\n"
    "- catalogue / log.error_count: count=25, p95=0.0\n"
    "- queue-master / log.row_count: count=25, p95=603.0\n"
    "(Many services have log.row_count and log.error_count entries; error_count p95 is 0 for the top rows.)\n\n"
    "Error logs: no error_log rows present.\n\n"
    "Notes: P95 values shown were computed on the full available series per (cmdb_id, metric) before any time-window filtering. Timestamps reported above are in UTC."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(        file_name  number_of_rows        earliest_timestamp          latest_timestamp  number_of_unique_cmdb_id
0      metric.csv            1900 2024-01-20 15:29:00+00:00 2024-01-20 15:53:00+00:00                        15
1       trace.csv               0                       NaT                       NaT                         0
2         log.csv             452 2024-01-20 15:29:00+00:00 2024-01-20 15:53:00+00:00                        10
3  error_logs.csv               0                       NaT                       NaT                         0,          cmdb_id    kpi_name  count_rows     p95_value
0          carts         cpu          25  2.522050e+00
1          carts  latency-50          25  1.976062e-02
2          carts  latency-90          25  4.346478e-02
3          carts         mem          25  2.137117e+08
4          carts      socket          25  1.200000e+01
5          carts    workload          25  9.001499e+00
6       carts-db         cpu          25  6.049812e+00
7       carts-db      diskio          25  2.436824e+06
8       carts-db         mem          25  9.294957e+07
9       carts-db      socket          25  7.000000e+00
10     catalogue         cpu          25  1.834278e-01
11     catalogue  latency-50          25  3.078649e-03
12     catalogue  latency-90          25  4.741568e-03
13     catalogue         mem          25  6.277775e+06
14     catalogue      socket          25  6.000000e+00
15     catalogue    workload          25  4.356848e+00
16  catalogue-db         cpu          25  2.143708e-01
17  catalogue-db      diskio          25  0.000000e+00
18  catalogue-db         mem          25  4.306460e+08
19  catalogue-db      socket          25  4.000000e+00, Empty DataFrame
Columns: [cmdb_id, trace_name, count_rows, p95_value]
Index: [],          cmdb_id         log_name  count_rows  p95_value
0      front-end    log.row_count          25     2074.2
1      front-end  log.error_count          25        0.0
2      catalogue    log.row_count          25      171.2
3      catalogue  log.error_count          25        0.0
4   queue-master  log.error_count          25        0.0
5   queue-master    log.row_count          25      603.0
6       shipping    log.row_count          25       67.0
7       shipping  log.error_count          25        0.0
8         orders  log.error_count          25        0.0
9         orders    log.row_count          25      134.0
10       payment  log.error_count          25        0.0
11       payment    log.row_count          25      107.0
12          user  log.error_count          25        0.0
13          user    log.row_count          25      620.4
14      carts-db  log.error_count          24        0.0
15      carts-db    log.row_count          24        1.0
16         carts  log.error_count          22        0.0
17         carts    log.row_count          22      147.5
18    session-db    log.row_count           5        5.0
19    session-db  log.error_count           5        0.0, Empty DataFrame
Columns: [cmdb_id, error_count]
Index: [])```
```

In [ ]:
# In[2]:

# Reuse previously loaded df_metric; ensure timestamps are parsed and present
# (df_metric was loaded in prior step and has 'timestamp' as UTC datetime)

import pandas as pd

selected_cmdb = ['carts', 'catalogue', 'orders', 'payment', 'user']

# Filter metric rows to only the selected components (keep full series for P95 calc)
df_metric_selected = df_metric[df_metric['cmdb_id'].isin(selected_cmdb)].copy()

# Ensure numeric values for reliable percentile calculations
df_metric_selected['num_value'] = pd.to_numeric(df_metric_selected['value'], errors='coerce')

# 2) Compute global P95 per (cmdb_id, kpi_name) using the full available series (no time filtering)
global_p95_df = (
    df_metric_selected
    .groupby(['cmdb_id', 'kpi_name'], dropna=False, as_index=False)
    .agg(global_p95=('num_value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')))
)

# 3) Filter metric.csv to the incident window (UTC inclusive)
window_start = pd.Timestamp("2024-01-20 15:26:24", tz="UTC")
window_end   = pd.Timestamp("2024-01-20 15:56:24", tz="UTC")
df_window = df_metric_selected[(df_metric_selected['timestamp'] >= window_start) & (df_metric_selected['timestamp'] <= window_end)].copy()

# 4) Merge window rows with global P95 and compute exceedances
merged = df_window.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Flag exceedances (value >= global_p95). Treat NaNs as non-exceeding.
merged['exceeds'] = merged['num_value'] >= merged['global_p95']
merged['exceeds'] = merged['exceeds'].fillna(False)

# helper column: timestamp only if exceed, else NaT (for earliest exceedance calc)
merged['timestamp_if_exceed'] = merged['timestamp'].where(merged['exceeds'], pd.NaT)

# Aggregate per (cmdb_id, kpi_name)
agg = (
    merged
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        global_p95 = ('global_p95', 'first'),
        count_points_in_window = ('timestamp', 'count'),
        count_points_exceeding_p95 = ('exceeds', 'sum'),
        earliest_timestamp_of_exceedance = ('timestamp_if_exceed', 'min'),
        max_value_in_window = ('num_value', 'max')
    )
)

# Ensure proper dtypes and ordering, and replace empty earliest timestamps with NaT
agg['earliest_timestamp_of_exceedance'] = pd.to_datetime(agg['earliest_timestamp_of_exceedance'], utc=True)

# 5) Sort and return up to top 20 rows by count_points_exceeding_p95 desc, then count_points_in_window desc
result_df = (
    agg
    .sort_values(['count_points_exceeding_p95', 'count_points_in_window'], ascending=[False, False])
    .reset_index(drop=True)
)

# Keep intermediate tables in kernel as requested: global_p95_df, df_window, merged, agg
global_p95_df, df_window, merged, agg, result_df.head(20)

```
Out[2]:
```
Summary of findings (UTC):

- Scope: analyzed metric.csv for components carts, catalogue, orders, payment, user. Global P95 per (cmdb_id, kpi_name) was computed from the full series (no time filtering). Then metrics were aggregated over the incident window 2024-01-20 15:26:24 → 2024-01-20 15:56:24 UTC.

Top results (sorted by count_points_exceeding_p95, then by count_points_in_window). Each row: cmdb_id | kpi_name | global_p95 | points_in_window | points_exceeding_p95 | earliest_exceedance (UTC) | max_value_in_window

1. catalogue | socket | 6.0 | 25 | 25 | 2024-01-20 15:29:00+00:00 | 6.0  
2. user      | error  | 0.0 | 25 | 25 | 2024-01-20 15:29:00+00:00 | 0.0  
3. user      | socket | 17.0| 25 | 25 | 2024-01-20 15:29:00+00:00 | 17.0  
4. carts     | socket | 12.0| 25 | 4  | 2024-01-20 15:34:00+00:00 | 12.43333...  
5. carts     | cpu    | 2.52205 | 25 | 2 | 2024-01-20 15:37:00+00:00 | 2.986773  
6. carts     | latency-50 | 0.01976062 | 25 | 2 | 2024-01-20 15:45:00+00:00 | 0.01981079  
7. carts     | latency-90 | 0.04346478 | 25 | 2 | 2024-01-20 15:46:00+00:00 | 0.04358099  
8. carts     | mem    | 2.137117e+08 | 25 | 2 | 2024-01-20 15:45:00+00:00 | 2.137432e+08  
9. carts     | workload | 9.001499 | 25 | 2 | 2024-01-20 15:41:00+00:00 | 9.056817  
10. catalogue | cpu   | 0.1834278 | 25 | 2 | 2024-01-20 15:42:00+00:00 | 0.1895385

Notes and interpretation:
- Many metrics have full coverage in the window (25 points → per-minute sampling).  
- Several (cmdb_id, kpi_name) pairs show all 25 points meeting or equaling the global P95 (e.g., catalogue/socket and user/socket). The aggregation used value >= global_p95, so rows equal to the P95 are counted as exceeding; this explains cases where p95 equals the observed max.  
- The most prominent signals to investigate first are socket-related metrics (catalogue.socket and user.socket) and carts.socket (multiple exceedances). Also inspect carts CPU/memory/latency and catalogue CPU/latency where some exceedances occurred.  
- The 'user/error' metric has global P95 = 0 and max = 0, so its 25 "exceedances" are due to value >= 0 (equality); treat that result with caution.

If you want, I can:
- show the full aggregated table (up to 20 rows) as a compact DataFrame, or  
- drill down to timestamps and raw values for any specific (cmdb_id, kpi_name) of interest.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id    kpi_name    global_p95
0    carts         cpu  2.522050e+00
1    carts  latency-50  1.976062e-02
2    carts  latency-90  4.346478e-02
3    carts         mem  2.137117e+08
4    carts      socket  1.200000e+01
..     ...         ...           ...
27    user  latency-50  4.985561e-03
28    user  latency-90  9.032135e-03
29    user         mem  8.448437e+06
30    user      socket  1.700000e+01
31    user    workload  2.003224e+01

[32 rows x 3 columns],                      timestamp cmdb_id    kpi_name         value    _num_value     num_value
0    2024-01-20 15:29:00+00:00   carts         cpu  1.505626e+00  1.505626e+00  1.505626e+00
1    2024-01-20 15:29:00+00:00   carts  latency-50  1.812719e-02  1.812719e-02  1.812719e-02
2    2024-01-20 15:29:00+00:00   carts  latency-90  4.013706e-02  4.013706e-02  4.013706e-02
3    2024-01-20 15:29:00+00:00   carts         mem  2.135654e+08  2.135654e+08  2.135654e+08
4    2024-01-20 15:29:00+00:00   carts      socket  1.100000e+01  1.100000e+01  1.100000e+01
...                        ...     ...         ...           ...           ...           ...
1891 2024-01-20 15:53:00+00:00    user  latency-50  4.715652e-03  4.715652e-03  4.715652e-03
1892 2024-01-20 15:53:00+00:00    user  latency-90  8.938269e-03  8.938269e-03  8.938269e-03
1893 2024-01-20 15:53:00+00:00    user         mem  8.455455e+06  8.455455e+06  8.455455e+06
1894 2024-01-20 15:53:00+00:00    user      socket  1.700000e+01  1.700000e+01  1.700000e+01
1895 2024-01-20 15:53:00+00:00    user    workload  1.928796e+01  1.928796e+01  1.928796e+01

[800 rows x 6 columns],                     timestamp cmdb_id    kpi_name         value    _num_value     num_value    global_p95  exceeds       timestamp_if_exceed
0   2024-01-20 15:29:00+00:00   carts         cpu  1.505626e+00  1.505626e+00  1.505626e+00  2.522050e+00    False                       NaT
1   2024-01-20 15:29:00+00:00   carts  latency-50  1.812719e-02  1.812719e-02  1.812719e-02  1.976062e-02    False                       NaT
2   2024-01-20 15:29:00+00:00   carts  latency-90  4.013706e-02  4.013706e-02  4.013706e-02  4.346478e-02    False                       NaT
3   2024-01-20 15:29:00+00:00   carts         mem  2.135654e+08  2.135654e+08  2.135654e+08  2.137117e+08    False                       NaT
4   2024-01-20 15:29:00+00:00   carts      socket  1.100000e+01  1.100000e+01  1.100000e+01  1.200000e+01    False                       NaT
..                        ...     ...         ...           ...           ...           ...           ...      ...                       ...
795 2024-01-20 15:53:00+00:00    user  latency-50  4.715652e-03  4.715652e-03  4.715652e-03  4.985561e-03    False                       NaT
796 2024-01-20 15:53:00+00:00    user  latency-90  8.938269e-03  8.938269e-03  8.938269e-03  9.032135e-03    False                       NaT
797 2024-01-20 15:53:00+00:00    user         mem  8.455455e+06  8.455455e+06  8.455455e+06  8.448437e+06     True 2024-01-20 15:53:00+00:00
798 2024-01-20 15:53:00+00:00    user      socket  1.700000e+01  1.700000e+01  1.700000e+01  1.700000e+01     True 2024-01-20 15:53:00+00:00
799 2024-01-20 15:53:00+00:00    user    workload  1.928796e+01  1.928796e+01  1.928796e+01  2.003224e+01    False                       NaT

[800 rows x 9 columns],    cmdb_id    kpi_name    global_p95  count_points_in_window  count_points_exceeding_p95 earliest_timestamp_of_exceedance  max_value_in_window
0    carts         cpu  2.522050e+00                      25                           2        2024-01-20 15:37:00+00:00         2.986773e+00
1    carts  latency-50  1.976062e-02                      25                           2        2024-01-20 15:45:00+00:00         1.981079e-02
2    carts  latency-90  4.346478e-02                      25                           2        2024-01-20 15:46:00+00:00         4.358099e-02
3    carts         mem  2.137117e+08                      25                           2        2024-01-20 15:45:00+00:00         2.137432e+08
4    carts      socket  1.200000e+01                      25                           4        2024-01-20 15:34:00+00:00         1.243333e+01
..     ...         ...           ...                     ...                         ...                              ...                  ...
27    user  latency-50  4.985561e-03                      25                           2        2024-01-20 15:33:00+00:00         5.006446e-03
28    user  latency-90  9.032135e-03                      25                           2        2024-01-20 15:44:00+00:00         9.071198e-03
29    user         mem  8.448437e+06                      25                           2        2024-01-20 15:52:00+00:00         8.455455e+06
30    user      socket  1.700000e+01                      25                          25        2024-01-20 15:29:00+00:00         1.700000e+01
31    user    workload  2.003224e+01                      25                           2        2024-01-20 15:29:00+00:00         2.029698e+01

[32 rows x 7 columns],       cmdb_id    kpi_name    global_p95  count_points_in_window  count_points_exceeding_p95 earliest_timestamp_of_exceedance  max_value_in_window
0   catalogue      socket  6.000000e+00                      25                          25        2024-01-20 15:29:00+00:00         6.000000e+00
1        user       error  0.000000e+00                      25                          25        2024-01-20 15:29:00+00:00         0.000000e+00
2        user      socket  1.700000e+01                      25                          25        2024-01-20 15:29:00+00:00         1.700000e+01
3       carts      socket  1.200000e+01                      25                           4        2024-01-20 15:34:00+00:00         1.243333e+01
4       carts         cpu  2.522050e+00                      25                           2        2024-01-20 15:37:00+00:00         2.986773e+00
5       carts  latency-50  1.976062e-02                      25                           2        2024-01-20 15:45:00+00:00         1.981079e-02
6       carts  latency-90  4.346478e-02                      25                           2        2024-01-20 15:46:00+00:00         4.358099e-02
7       carts         mem  2.137117e+08                      25                           2        2024-01-20 15:45:00+00:00         2.137432e+08
8       carts    workload  9.001499e+00                      25                           2        2024-01-20 15:41:00+00:00         9.056817e+00
9   catalogue         cpu  1.834278e-01                      25                           2        2024-01-20 15:42:00+00:00         1.895385e-01
10  catalogue  latency-50  3.078649e-03                      25                           2        2024-01-20 15:50:00+00:00         3.105893e-03
11  catalogue  latency-90  4.741568e-03                      25                           2        2024-01-20 15:50:00+00:00         4.790608e-03
12  catalogue         mem  6.277775e+06                      25                           2        2024-01-20 15:51:00+00:00         6.287360e+06
13  catalogue    workload  4.356848e+00                      25                           2        2024-01-20 15:41:00+00:00         4.440950e+00
14     orders         cpu  2.610416e+00                      25                           2        2024-01-20 15:36:00+00:00         4.527504e+00
15     orders  latency-50  7.141559e-02                      25                           2        2024-01-20 15:45:00+00:00         7.292841e-02
16     orders  latency-90  1.987531e-01                      25                           2        2024-01-20 15:45:00+00:00         1.993862e-01
17     orders         mem  3.274873e+08                      25                           2        2024-01-20 15:41:00+00:00         3.275414e+08
18     orders      socket  1.552000e+01                      25                           2        2024-01-20 15:41:00+00:00         1.621667e+01
19     orders    workload  2.218207e+00                      25                           2        2024-01-20 15:29:00+00:00         2.255667e+00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse selected_cmdb and df_log from previous kernel state (they exist)
# If not, define selected_cmdb here
try:
    selected_cmdb
except NameError:
    selected_cmdb = ['carts', 'catalogue', 'orders', 'payment', 'user']

# Ensure df_log is loaded and timestamps parsed (from earlier steps they should be)
# But guard in case: reload/parse minimally if needed
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
else:
    if df_log['timestamp'].dtype != 'datetime64[ns, UTC]':
        df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# 1) Restrict to selected components
df_log_selected = df_log[df_log['cmdb_id'].isin(selected_cmdb)].copy()

# 2) Compute global P95 per (cmdb_id, log_name) using full series (no time filtering)
df_log_selected['num_value'] = pd.to_numeric(df_log_selected['value'], errors='coerce')
global_p95_log = (
    df_log_selected
    .groupby(['cmdb_id', 'log_name'], dropna=False, as_index=False)
    .agg(global_p95=('num_value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')))
)

# 3) Filter log rows to incident window (UTC inclusive)
window_start = pd.Timestamp("2024-01-20 15:26:24", tz="UTC")
window_end   = pd.Timestamp("2024-01-20 15:56:24", tz="UTC")
df_log_window = df_log_selected[(df_log_selected['timestamp'] >= window_start) & (df_log_selected['timestamp'] <= window_end)].copy()

# 4) Merge window rows with global P95 and compute exceedances (value >= global_p95)
merged_log = df_log_window.merge(global_p95_log, on=['cmdb_id', 'log_name'], how='left')
merged_log['exceeds'] = merged_log['num_value'] >= merged_log['global_p95']
merged_log['exceeds'] = merged_log['exceeds'].fillna(False)
merged_log['timestamp_if_exceed'] = merged_log['timestamp'].where(merged_log['exceeds'], pd.NaT)

agg_log = (
    merged_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        global_p95 = ('global_p95', 'first'),
        count_points_in_window = ('timestamp', 'count'),
        count_points_exceeding_p95 = ('exceeds', 'sum'),
        earliest_timestamp_of_exceedance = ('timestamp_if_exceed', 'min'),
        max_value_in_window = ('num_value', 'max')
    )
)

# Ensure earliest timestamps are timezone-aware datetimes (or NaT)
agg_log['earliest_timestamp_of_exceedance'] = pd.to_datetime(agg_log['earliest_timestamp_of_exceedance'], utc=True)

# 5) Filter to rows with count_points_exceeding_p95 > 0 and sort
result_log = (
    agg_log[agg_log['count_points_exceeding_p95'] > 0]
    .sort_values(['count_points_exceeding_p95', 'count_points_in_window'], ascending=[False, False])
    .reset_index(drop=True)
)

# Keep intermediates in kernel: df_log_selected, global_p95_log, df_log_window, merged_log, agg_log
global_p95_log, df_log_window, merged_log, agg_log, result_log.head(20)

```
Out[3]:
```
summary = (
    "Log metrics summary (UTC) for components [carts, catalogue, orders, payment, user]\n"
    "Incident window: 2024-01-20 15:26:24 → 2024-01-20 15:56:24 UTC\n\n"
    "Top (cmdb_id, log_name) rows with any points >= global P95 (sorted by exceedances):\n"
    "1) catalogue | log.error_count | global_p95=0.0 | points_in_window=25 | exceedances=25 | first_exceed=2024-01-20 15:29:00 | max=0\n"
    "2) orders    | log.error_count | global_p95=0.0 | points_in_window=25 | exceedances=25 | first_exceed=2024-01-20 15:29:00 | max=0\n"
    "3) payment   | log.error_count | global_p95=0.0 | points_in_window=25 | exceedances=25 | first_exceed=2024-01-20 15:29:00 | max=0\n"
    "4) user      | log.error_count | global_p95=0.0 | points_in_window=25 | exceedances=25 | first_exceed=2024-01-20 15:29:00 | max=0\n"
    "5) carts     | log.error_count | global_p95=0.0 | points_in_window=22 | exceedances=22 | first_exceed=2024-01-20 15:30:00 | max=0\n"
    "6) orders    | log.row_count   | global_p95=134.0 | points_in_window=25 | exceedances=3  | first_exceed=2024-01-20 15:36:00 | max=140\n"
    "7) payment   | log.row_count   | global_p95=107.0 | points_in_window=25 | exceedances=3  | first_exceed=2024-01-20 15:36:00 | max=110\n"
    "8) catalogue | log.row_count   | global_p95=171.2 | points_in_window=25 | exceedances=2  | first_exceed=2024-01-20 15:34:00 | max=179\n"
    "9) user      | log.row_count   | global_p95=620.4 | points_in_window=25 | exceedances=2  | first_exceed=2024-01-20 15:41:00 | max=672\n"
    "10) carts    | log.row_count   | global_p95=147.5 | points_in_window=22 | exceedances=2  | first_exceed=2024-01-20 15:41:00 | max=175\n\n"
    "Interpretation and notes:\n"
    "- Many 'log.error_count' entries show global P95 = 0 and max = 0; these appear as 'exceedances' because we counted value >= P95 (0 >= 0). This is not indicative of real errors — treat those rows as non-actionable.\n"
    "- Real signals to inspect: log.row_count spikes. Orders and Payment each had 3 points above their P95 (first at 15:36 UTC); catalogue, user, and carts had 2 exceedances each at the times shown. These may indicate short traffic spikes or bursts of log activity.\n"
    "- Recommended next steps: examine raw log_row values and surrounding timestamps for the log.row_count exceedances (particularly orders/payment at ~15:36 UTC) and correlate with application/metric traces for load or latency changes.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id         log_name  global_p95
0      carts  log.error_count         0.0
1      carts    log.row_count       147.5
2  catalogue  log.error_count         0.0
3  catalogue    log.row_count       171.2
4     orders  log.error_count         0.0
5     orders    log.row_count       134.0
6    payment  log.error_count         0.0
7    payment    log.row_count       107.0
8       user  log.error_count         0.0
9       user    log.row_count       620.4,                     timestamp    cmdb_id         log_name  value  _num_value  num_value
0   2024-01-20 15:29:00+00:00  catalogue  log.error_count      0           0          0
1   2024-01-20 15:29:00+00:00  catalogue    log.row_count     97          97         97
4   2024-01-20 15:29:00+00:00     orders  log.error_count      0           0          0
5   2024-01-20 15:29:00+00:00     orders    log.row_count     86          86         86
6   2024-01-20 15:29:00+00:00    payment  log.error_count      0           0          0
..                        ...        ...              ...    ...         ...        ...
443 2024-01-20 15:53:00+00:00     orders    log.row_count     56          56         56
444 2024-01-20 15:53:00+00:00    payment  log.error_count      0           0          0
445 2024-01-20 15:53:00+00:00    payment    log.row_count     44          44         44
450 2024-01-20 15:53:00+00:00       user  log.error_count      0           0          0
451 2024-01-20 15:53:00+00:00       user    log.row_count    261         261        261

[244 rows x 6 columns],                     timestamp    cmdb_id         log_name  value  _num_value  num_value  global_p95  exceeds       timestamp_if_exceed
0   2024-01-20 15:29:00+00:00  catalogue  log.error_count      0           0          0         0.0     True 2024-01-20 15:29:00+00:00
1   2024-01-20 15:29:00+00:00  catalogue    log.row_count     97          97         97       171.2    False                       NaT
2   2024-01-20 15:29:00+00:00     orders  log.error_count      0           0          0         0.0     True 2024-01-20 15:29:00+00:00
3   2024-01-20 15:29:00+00:00     orders    log.row_count     86          86         86       134.0    False                       NaT
4   2024-01-20 15:29:00+00:00    payment  log.error_count      0           0          0         0.0     True 2024-01-20 15:29:00+00:00
..                        ...        ...              ...    ...         ...        ...         ...      ...                       ...
239 2024-01-20 15:53:00+00:00     orders    log.row_count     56          56         56       134.0    False                       NaT
240 2024-01-20 15:53:00+00:00    payment  log.error_count      0           0          0         0.0     True 2024-01-20 15:53:00+00:00
241 2024-01-20 15:53:00+00:00    payment    log.row_count     44          44         44       107.0    False                       NaT
242 2024-01-20 15:53:00+00:00       user  log.error_count      0           0          0         0.0     True 2024-01-20 15:53:00+00:00
243 2024-01-20 15:53:00+00:00       user    log.row_count    261         261        261       620.4    False                       NaT

[244 rows x 9 columns],      cmdb_id         log_name  global_p95  count_points_in_window  count_points_exceeding_p95 earliest_timestamp_of_exceedance  max_value_in_window
0      carts  log.error_count         0.0                      22                          22        2024-01-20 15:30:00+00:00                    0
1      carts    log.row_count       147.5                      22                           2        2024-01-20 15:41:00+00:00                  175
2  catalogue  log.error_count         0.0                      25                          25        2024-01-20 15:29:00+00:00                    0
3  catalogue    log.row_count       171.2                      25                           2        2024-01-20 15:34:00+00:00                  179
4     orders  log.error_count         0.0                      25                          25        2024-01-20 15:29:00+00:00                    0
5     orders    log.row_count       134.0                      25                           3        2024-01-20 15:36:00+00:00                  140
6    payment  log.error_count         0.0                      25                          25        2024-01-20 15:29:00+00:00                    0
7    payment    log.row_count       107.0                      25                           3        2024-01-20 15:36:00+00:00                  110
8       user  log.error_count         0.0                      25                          25        2024-01-20 15:29:00+00:00                    0
9       user    log.row_count       620.4                      25                           2        2024-01-20 15:41:00+00:00                  672,      cmdb_id         log_name  global_p95  count_points_in_window  count_points_exceeding_p95 earliest_timestamp_of_exceedance  max_value_in_window
0  catalogue  log.error_count         0.0                      25                          25        2024-01-20 15:29:00+00:00                    0
1     orders  log.error_count         0.0                      25                          25        2024-01-20 15:29:00+00:00                    0
2    payment  log.error_count         0.0                      25                          25        2024-01-20 15:29:00+00:00                    0
3       user  log.error_count         0.0                      25                          25        2024-01-20 15:29:00+00:00                    0
4      carts  log.error_count         0.0                      22                          22        2024-01-20 15:30:00+00:00                    0
5     orders    log.row_count       134.0                      25                           3        2024-01-20 15:36:00+00:00                  140
6    payment    log.row_count       107.0                      25                           3        2024-01-20 15:36:00+00:00                  110
7  catalogue    log.row_count       171.2                      25                           2        2024-01-20 15:34:00+00:00                  179
8       user    log.row_count       620.4                      25                           2        2024-01-20 15:41:00+00:00                  672
9      carts    log.row_count       147.5                      22                           2        2024-01-20 15:41:00+00:00                  175)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```